In [1]:
import torch, math
from torch import nn
from torch.nn import functional as F

In [2]:
torch.manual_seed(1337)

batch_size = 64
block_size = 256 # max context length
n_embd = 384
lr = 3e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200
eval_interval = 500
max_iters = 5000
dropout = 0.2
n_dlayers = 6
n_elayers = 6
n_heads = 6

In [3]:
with open(f"data/dialogues_train.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = text.replace("__eou__ ", "")

chars = sorted(list(set(text)))
vocab_size = len(chars)

s_i = { ch : i for i, ch in enumerate(chars) }
i_s = { i : ch for i, ch in enumerate(chars) }

encode = lambda s: [s_i[c] for c in s]
decode = lambda l: "".join([i_s[i] for i in l])

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.8*len(data))

dtr = data[:n]
dval = data[n:]

def get_batch(split):
    data = dtr if split == "train" else dval
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i+1 : i+block_size+1] for i in ix])

    x, y = x.to(device), y.to(device)

    return x, y

In [19]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V):
        B, T, C = Q.shape
        k = self.key(K)
        q = self.query(Q)

        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei)

        v = self.value(V)
        out = wei @ v

        return out

In [20]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList((Head(head_size) for _ in range(num_heads)))
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V):
        out = torch.cat([h(Q, K, V) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))

        return out

In [21]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )


    def forward(self, x):
        return self.net(x)

In [22]:
class Encoder(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.s_attn = MultiHeadAttention(n_head, head_size)
        self.f_fwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = self.ln1(x)
        x = x + self.s_attn(x, x, x)

        x = self.ln2(x)
        x = x + self.f_fwd(x)

        return x

In [23]:
class Decoder(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.s_attn = MultiHeadAttention(n_head, head_size)
        self.c_attn = MultiHeadAttention(n_head, head_size)
        self.f_fwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, enc_output):
        x = self.ln1(x)
        x = x + self.s_attn(x, x, x)

        x = self.ln2(x)
        x = x + self.c_attn(x, enc_output, enc_output)

        x = self.ln3(x)
        x = x + self.f_fwd(x)

        return x

In [24]:
class PositionalEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(block_size, n_embd)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return x + self.embedding(positions).unsqueeze(0)

In [25]:
class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_embd = nn.Embedding(vocab_size, n_embd)
        self.dec_embd = nn.Embedding(vocab_size, n_embd)
        self.pos_embd = PositionalEmbedding()

        self.encoders = nn.Sequential(*[Encoder(n_embd, n_heads) for _ in range(n_elayers)])
        self.decoders = nn.ModuleList([Decoder(n_embd, n_heads) for _ in range(n_dlayers)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        enc_emb = self.pos_embd(self.enc_embd(idx))
        enc_output = self.encoders(enc_emb)

        x = self.pos_embd(self.dec_embd(idx))
        for decoder in self.decoders:
            x = decoder(x, enc_output)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(-1)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [6]:
import gc

if 'text' in globals():
    del context

gc.collect()

torch.cuda.empty_cache()

In [26]:
net = Transformer().to(device)
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3)

In [39]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [29]:
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(net.generate(context, max_new_tokens=100)[0].tolist()))


fmE¥YH~。Rd¥mI’)GY8KNkITNk¥(”G,'I .%(’5x2B°t °\W;/s'fY£
*R‘w!Q。¥v—=+ib!I'isQkY:5A3-@G,、4J;+2£R
$8Z
~


In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    net.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, y = get_batch(split)
            logits, loss = net(X, y)
            losses[k] = loss.item()

        out[split] = losses.mean()

    net.train()
    return out

In [ ]:
estimate_loss()

In [ ]:
for step in range(max_iters):
    Xb, yb = get_batch("train")

    logits, loss = net(Xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        print(f'{step} / {max_iters}: {loss.item()}')

print("Final loss:", loss)